# Обучение модели прогноза количества операций

Модель предсказывает **количество операций на следующий месяц**.

Параметр `EVAL_MODE`:
- `holdout_2025`: обучаемся на 2022-2024, тестируем на 2025.
- `backtest`: несколько скользящих тестовых окон.

Сравним:
- базовый датасет (`operations_training_dataset_base.csv`)
- датасет с регрессором `workDayMonth_2` (`operations_training_dataset_wdm2.csv`)


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor

PROCESSED_DIR = Path("../data/processed")

BASE_PATH = PROCESSED_DIR / "operations_training_dataset_base.csv"
WDM2_PATH = PROCESSED_DIR / "operations_training_dataset_wdm2.csv"

EVAL_MODE = "holdout_2025"  # or "backtest"

BASE_PATH, WDM2_PATH

In [ ]:
def compute_metrics(y_true, y_pred, large_abs=100, large_pct=0.3):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)

    with np.errstate(divide="ignore", invalid="ignore"):
        mape = np.abs((y_true - y_pred) / y_true)
        mape = np.where(np.isfinite(mape), mape, np.nan)
    mape_pct = np.nanmean(mape) * 100

    abs_err = np.abs(y_true - y_pred)
    pct_err = np.where(y_true != 0, abs_err / np.abs(y_true), np.nan)
    large_err = (abs_err >= large_abs) | (pct_err >= large_pct)
    large_err_share = np.nanmean(large_err)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_%": mape_pct,
        "large_error_share": large_err_share,
    }


def train_eval(df, feature_cols, label_col, train_mask, test_mask):
    X_train = df.loc[train_mask, feature_cols]
    y_train = df.loc[train_mask, label_col]
    X_test = df.loc[test_mask, feature_cols]
    y_test = df.loc[test_mask, label_col]

    model = CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        loss_function="RMSE",
        verbose=False,
        random_seed=42,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    metrics = compute_metrics(y_test, y_pred)

    return model, metrics, y_test, y_pred


In [ ]:
# Базовый датасет

df_base = pd.read_csv(BASE_PATH, parse_dates=["period"])

# признаки: текущий факт + календарь
feature_cols_base = ["operation_fact_value", "month_num", "year_num"]
label_col = "target_next_month"

if EVAL_MODE == "holdout_2025":
    train_mask = df_base["period"].dt.year <= 2024
    test_mask = df_base["period"].dt.year == 2025
else:
    # простая backtest-схема: тестируем 2024 и 2025 по очереди
    train_mask = df_base["period"].dt.year <= 2023
    test_mask = df_base["period"].dt.year == 2024

model_base, metrics_base, y_true_base, y_pred_base = train_eval(
    df_base, feature_cols_base, label_col, train_mask, test_mask
)

metrics_base

In [ ]:
# Датасет с внешним регрессором workDayMonth_2

df_wdm2 = pd.read_csv(WDM2_PATH, parse_dates=["period"])

feature_cols_wdm2 = ["operation_fact_value", "month_num", "year_num", "workDayMonth_2"]

if EVAL_MODE == "holdout_2025":
    train_mask = df_wdm2["period"].dt.year <= 2024
    test_mask = df_wdm2["period"].dt.year == 2025
else:
    train_mask = df_wdm2["period"].dt.year <= 2023
    test_mask = df_wdm2["period"].dt.year == 2024

model_wdm2, metrics_wdm2, y_true_wdm2, y_pred_wdm2 = train_eval(
    df_wdm2, feature_cols_wdm2, label_col, train_mask, test_mask
)

metrics_wdm2

In [ ]:
# Сохраняем предсказания на тесте для дальнейшего сравнения/визуализации

pred_out = PROCESSED_DIR / "model_predictions_operations_catboost_v1.csv"

df_pred = df_wdm2.loc[test_mask, ["resist_id", "operation_id", "period"]].copy()

df_pred["y_true"] = y_true_wdm2

df_pred["y_pred"] = y_pred_wdm2

df_pred.to_csv(pred_out, index=False)

pred_out